# Compass Gate 1 — Baseline SLM Evaluation

This notebook evaluates the untrained Qwen3-1.7B model against Compass's held-out synthetic benchmark. It measures latency, tool choice, argument validity, clarification behavior, and possible numeric invention. Do not upload real financial data.

In [ ]:
%pip install -q -U 'transformers>=4.57.1' 'accelerate>=1.10.1' 'safetensors>=0.8.0'

In [ ]:
from google.colab import files
uploaded = files.upload()
benchmark_name = next((name for name in uploaded if name.startswith('benchmark_test') and name.endswith('.jsonl')), None)
assert benchmark_name is not None, 'Upload benchmark_test.jsonl from the Compass model folder.'
print(f'Loaded {benchmark_name}: {len(uploaded[benchmark_name].splitlines())} cases')

In [ ]:
import json
import re
import time
from pathlib import Path

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen3-1.7B'
APPROVED_TOOLS = {'get_income', 'get_expenses', 'get_category_spending', 'get_largest_expenses', 'get_surplus'}
rows = [json.loads(line) for line in uploaded[benchmark_name].decode().splitlines() if line.strip()]
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
print(f'Model: {MODEL_ID}; cases: {len(rows)}; device: {model.device}')

In [ ]:
SYSTEM = 'You are the untrained baseline for Compass. Return only one JSON object. Use approved tools only: get_income, get_expenses, get_category_spending, get_largest_expenses, or get_surplus. For unsupported or unclear questions, return a JSON object with tool null and needs_clarification true. Never calculate, guess, or state a financial amount. Never invent evidence.'


In [ ]:
def parse_json(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, re.S)
        if not match:
            return None
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return None

def evaluate_output(item, text, latency):
    parsed = parse_json(text)
    expected_tool = item['expected_tool']
    tool = parsed.get('tool') if isinstance(parsed, dict) else None
    arguments = (parsed.get('arguments') or parsed.get('parameters') or {}) if isinstance(parsed, dict) else {}
    valid_tool = tool is None or tool in APPROVED_TOOLS
    date_ok = isinstance(arguments, dict) and ('month' in arguments or 'date' in arguments)
    limit_ok = tool != 'get_largest_expenses' or isinstance(arguments.get('limit'), int) and 1 <= arguments['limit'] <= 20
    category_ok = tool != 'get_category_spending' or isinstance(arguments.get('category'), str) and bool(arguments['category'].strip())
    valid_args = tool is None or (isinstance(arguments, dict) and date_ok and limit_ok and category_ok)
    clarification = bool(parsed and parsed.get('needs_clarification')) if isinstance(parsed, dict) else False
    numeric_claim = bool(re.search(r'(?:\$|\b\d+[.,]\d{2}\b)', text))
    return {
        'question': item['question'],
        'expected_intent': item['expected_intent'],
        'expected_tool': expected_tool,
        'scenario': item['scenario'],
        'raw_response': text,
        'parsed_response': parsed,
        'selected_tool': tool,
        'tool_match': tool == expected_tool,
        'valid_tool': valid_tool,
        'valid_arguments': valid_args,
        'appropriate_clarification': (expected_tool is None and clarification and tool is None) or (expected_tool is not None and not clarification),
        'possible_numeric_invention': numeric_claim,
        'latency_seconds': round(latency, 3),
    }

In [ ]:
results = []
for number, item in enumerate(rows, 1):
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': item['question']},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    started = time.perf_counter()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=96, do_sample=False)
    latency = time.perf_counter() - started
    generated = output[0][inputs['input_ids'].shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()
    results.append(evaluate_output(item, text, latency))
    if number % 10 == 0:
        print(f'Completed {number}/{len(rows)}')

Path('baseline_results.jsonl').write_text(''.join(json.dumps(row, sort_keys=True) + '\n' for row in results))
print('Saved baseline_results.jsonl')

In [ ]:
summary = {
    'model_id': MODEL_ID,
    'cases': len(results),
    'tool_match_rate': round(sum(row['tool_match'] for row in results) / len(results), 3),
    'valid_tool_rate': round(sum(row['valid_tool'] for row in results) / len(results), 3),
    'valid_argument_rate': round(sum(row['valid_arguments'] for row in results) / len(results), 3),
    'appropriate_clarification_rate': round(sum(row['appropriate_clarification'] for row in results) / len(results), 3),
    'possible_numeric_invention_count': sum(row['possible_numeric_invention'] for row in results),
    'mean_latency_seconds': round(sum(row['latency_seconds'] for row in results) / len(results), 3),
    'synthetic_data_only': True,
}
Path('baseline_report.json').write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))

In [ ]:
from google.colab import files
files.download('baseline_results.jsonl')
files.download('baseline_report.json')